# 몽글마을 LLM 모델 평가 노트북

이 노트북은 `docs/model-evaluation-plan-v2.md`, `docs/model-evaluation-rubric.md`, `docs/codex-llm-evaluation-notebook-prompt.md` 기준으로 오픈소스 LLM 후보 모델을 비교합니다.

- 실제 사용자 데이터는 사용하지 않고, `data/generated`의 합성 평가 데이터만 사용합니다.
- 기본값은 `RUN_MOCK_ONLY=True`라서 GPU/API 없이도 전체 저장·평가 파이프라인을 확인할 수 있습니다.
- RunPod에서 실제 모델을 돌릴 때는 `RUN_MOCK_ONLY=False`로 바꾸고 필요한 모델의 `enabled` 값을 조정하세요.
- GPT-5.5 심판 평가는 비용이 생길 수 있어 `JUDGE_CONFIG["enabled"] = False`가 기본값입니다.

## 1. 설치 안내

RunPod 새 환경에서 필요한 패키지가 없다면 아래 셀의 주석을 해제해 설치하세요.

In [1]:

# 필요할 때만 주석을 해제해서 실행하세요.
# !pip install -U transformers accelerate torch pandas python-dotenv openai tqdm matplotlib


## 2. Imports

In [2]:

import csv
import gc
import json
import os
import re
import time
import traceback
import uuid
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None


## 3-2. torchvision 우회 (텍스트 전용 평가)
팟의 torchvision이 torch와 버전 불일치로 깨져 있으면 transformers가 모델 로드 시 크래시한다. 텍스트 LLM 평가엔 불필요하므로 transformers가 import하기 전에 제거한다.

In [3]:
# torchvision이 torch와 안 맞아 깨진 경우, transformers가 모델 로드 시 무조건 import해서 크래시함.
# 텍스트 LLM 평가엔 torchvision이 필요 없으므로 transformers가 건드리기 전에 제거한다.
import sys, importlib.util, subprocess

try:
    import torch
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
except Exception as exc:
    print("torch import 실패:", exc)

if "torchvision" in sys.modules:
    print("\u26a0\ufe0f torchvision이 이미 import됨 \u2192 Kernel > Restart Kernel 후 이 셀부터 다시 실행하세요.")
elif importlib.util.find_spec("torchvision") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
    importlib.invalidate_caches()
    remaining = importlib.util.find_spec("torchvision")
    print("\u2705 torchvision 제거" + ("" if remaining is None else " (아직 남음: 재시작 필요)"))
else:
    print("\u2705 torchvision 없음 \u2192 진행 가능")


torch 2.13.0+cu130 cuda True
✅ torchvision 없음 → 진행 가능


## 3-3. HF 모델 캐시를 큰 볼륨으로
컨테이너 디스크(`/`)는 작아서 모델을 여러 개 받으면 꽉 찬다(`No space left on device`). RunPod 볼륨 `/workspace`가 있으면 그쪽으로 캐시를 돌린다. transformers/HF import 전에 실행해야 한다.

In [4]:
# HF 모델 캐시를 큰 볼륨으로 이동. 컨테이너 디스크(/)는 보통 작아서(50GB 남짓)
# 모델 여러 개 받으면 "No space left on device"가 난다. /workspace(RunPod 볼륨)로 돌린다.
# 주의: 이 셀은 transformers/huggingface_hub 를 처음 import 하는 셀보다 먼저 실행돼야 한다.
import os
_hf_vol = "/workspace"
if os.path.isdir(_hf_vol):
    os.environ["HF_HOME"] = f"{_hf_vol}/hf-cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print("HF_HOME =", os.environ["HF_HOME"])
else:
    print("HF_HOME 기본값 사용 (/workspace 없음)")


HF_HOME = /workspace/hf-cache


## 3. Path / Environment 설정

In [5]:

# 노트북을 프로젝트 루트 또는 notebooks/ 안에서 실행해도 동작하도록 루트를 찾습니다.
def find_project_root() -> Path:
    # 데이터가 있는 폴더(llm_evaluation)를 기준으로 삼는다.
    # 노트북은 notebooks/ 하위에서 실행되므로 위로 올라가며 data/generated 를 찾는다.
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "data" / "generated").is_dir():
            return path
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
GENERATED_DIR = DATA_DIR / "generated"
FIXTURE_DIR = DATA_DIR / "fixtures"
EVAL_DIR = DATA_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

if load_dotenv:
    load_dotenv(PROJECT_ROOT / ".env")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]

ENV = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY", ""),
    "HF_TOKEN": os.getenv("HF_TOKEN", ""),
    "VLLM_BASE_URL": os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1"),
    "VLLM_API_KEY": os.getenv("VLLM_API_KEY", "EMPTY"),
}

# True면 실제 모델 대신 MockProvider로 빠르게 전체 파이프라인을 점검합니다.
# RunPod에서 실제 모델 평가를 실행하려면 False로 바꾸세요.
RUN_MOCK_ONLY = False
MAX_SAMPLES_PER_TASK = 20  # 전체 평가 시 None으로 바꾸세요.
APPEND_OUTPUTS = False
CLEAR_CUDA_AFTER_MODEL = True

# 속도 점수 기준입니다. RTX 5090에서 4B~8B 모델을 단일 샘플 생성으로 평가하는 초기값입니다.
# 기능별 출력 길이가 달라서 planner는 더 여유 있는 기준을 씁니다.
SPEED_THRESHOLDS = {
    "todo_decomposition": {"fast": 2.0, "slow": 15.0},
    "planner_chatbot": {"fast": 5.0, "slow": 45.0},
    "quest_generation": {"fast": 2.0, "slow": 15.0},
    "feed_post_generation": {"fast": 2.0, "slow": 15.0},
    "character_message_generation": {"fast": 2.0, "slow": 15.0},
}
DEFAULT_SPEED_THRESHOLD = {"fast": 2.0, "slow": 30.0}

# RTX 5090 단일 GPU에서는 4B~8B 모델을 CPU offload 없이 GPU에 올리는 편이 빠릅니다.
# 메모리 부족이 나면 False로 바꾸면 됩니다.
FORCE_SINGLE_GPU = True
PRINT_MODEL_PLACEMENT = True

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUN_ID: {RUN_ID}")
print("OPENAI_API_KEY:", "있음" if ENV["OPENAI_API_KEY"] else "없음")
print("HF_TOKEN:", "있음" if ENV["HF_TOKEN"] else "없음")


PROJECT_ROOT: /mongle-ai/llm_evaluation
RUN_ID: 20260709_165844_4dc34de4
OPENAI_API_KEY: 있음
HF_TOKEN: 있음


## 4. Model Config

In [6]:
MODEL_CONFIGS = {
    "midm-mini": {
        "provider": "transformers",
        "model_id": "K-intelligence/Midm-2.0-Mini-Instruct",
        "enabled": True,
    },
    "midm-base": {
        "provider": "transformers",
        "model_id": "K-intelligence/Midm-2.0-Base-Instruct",
        "enabled": False,
    },
    "qwen-7b": {
        "provider": "transformers",
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "enabled": True,
    },
    "ministral-8b": {
        "provider": "transformers",
        "model_id": "mistralai/Ministral-3-8B-Instruct-2512-BF16",
        "enabled": False,
        "tokenizer_kwargs": {"fix_mistral_regex": True},
    },
    "llama-8b": {
        "provider": "transformers",
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "enabled": False,
    },
    "exaone-7.8b": {
        "provider": "transformers",
        "model_id": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
        "enabled": True,
    },
    "gemma-3-4b-it": {
        "provider": "transformers",
        "model_id": "google/gemma-3-4b-it",
        "enabled": False,
    },
    "phi-4-mini": {
        "provider": "transformers",
        "model_id": "microsoft/Phi-4-mini-instruct",
        "enabled": False,
    },
    # vLLM 서버를 띄운 경우 예시입니다. 필요할 때 enabled를 켜세요.
    "local-vllm": {
        "provider": "openai_compatible",
        "model_id": "local-model",
        "enabled": False,
    },
}

JUDGE_CONFIG = {
    "enabled": True,
    "provider": "openai",
    "model_id": "gpt-5.5",
    # OpenAI 최신 모델은 Responses API를 우선 사용합니다.
    # 필요하면 "chat_completions"로 바꿔서 이전 방식도 테스트할 수 있습니다.
    "api": "responses",
    "temperature": 0.0,
    "max_output_tokens": 2048,
}

# HF 토큰이 없으면 보통 접근 승인이 필요한 모델은 건너뜁니다.
GATED_MODEL_PREFIXES = ("meta-llama/", "mistralai/", "google/")


## 5. Generation Config

In [7]:

GENERATION_CONFIG = {
    "temperature": 0.3,
    "top_p": 0.9,
    "max_new_tokens": 512,
    "repetition_penalty": 1.05,
    "do_sample": True,
}

# 구조화 JSON 태스크는 짧게 끝나야 하므로 max_new_tokens를 작게 둡니다.
# 모델이 EOS를 잘 내지 못하면 이 값이 샘플당 속도를 크게 좌우합니다.
MODEL_OVERRIDES = {
    "todo_decomposition": {"temperature": 0.1, "max_new_tokens": 128, "do_sample": False},
    "planner_chatbot": {"temperature": 0.2, "max_new_tokens": 512, "do_sample": False},
    "quest_generation": {"temperature": 0.7, "max_new_tokens": 64, "do_sample": True},
    "feed_post_generation": {"temperature": 0.6, "max_new_tokens": 96, "do_sample": True},
    "character_message_generation": {"temperature": 0.6, "max_new_tokens": 96, "do_sample": True},
}

def generation_config_for(task: str) -> Dict[str, Any]:
    config = dict(GENERATION_CONFIG)
    config.update(MODEL_OVERRIDES.get(task, {}))
    return config


## 6. Prompt Templates

In [8]:

SERVICE_TONE = """
[서비스 톤]
- 한국어로만 응답한다.
- 따뜻하고 다정하지만 과하게 유치하지 않다.
- 사용자를 비난하거나 압박하지 않는다.
- 폭력적, 선정적, 혐오적, 위협적 표현을 쓰지 않는다.
- 캐릭터는 AI 비서가 아니라 몽글마을에 사는 작은 주민처럼 말한다.
""".strip()

PROMPT_TEMPLATES = {
    "todo_decomposition": f"""{SERVICE_TONE}

너는 몽글마을의 TODO 자동 생성기다.
사용자 문장을 실행 가능한 TODO 단위로 나누어라.

규칙:
- JSON으로만 응답한다. 마크다운 코드블록은 쓰지 않는다.
- 출력 스키마: {{"todos": [{{"title": string, "tag": string, "date": string|null, "time": string|null}}]}}
- TODO가 아닌 감정 표현이나 배경 설명은 제외한다.
- 제목은 짧고 자연스러운 한국어 동사형으로 쓴다.
- 빈 TODO와 중복 TODO를 만들지 않는다.

[입력]
{{input_json}}
""",
    "planner_chatbot": f"""{SERVICE_TONE}

너는 몽글마을의 플래너 챗봇이다.
사용자의 일정 요청을 보고 추가 질문 또는 날짜별 플랜 중 하나를 만든다.

규칙:
- JSON으로만 응답한다. 마크다운 코드블록은 쓰지 않는다.
- 정보가 부족하면 response_type은 "clarifying_question"으로 한다.
- 정보가 충분하면 response_type은 "plan"으로 하고 todos와 calendar_events를 만든다.
- 답변은 전체 1500자 이내를 목표로 한다.
- 마감일이 짧으면 현실적이고 핵심적인 계획을 우선한다.
- 입력에 없는 시험명, 과목, 제약을 과도하게 지어내지 않는다.

clarifying_question 스키마:
{{"response_type":"clarifying_question","message":string,"missing_fields":[string]}}

plan 스키마:
{{"response_type":"plan","summary":string,"todos":[{{"date":"YYYY-MM-DD","title":string,"tag":string,"estimated_minutes":number}}],"calendar_events":[{{"date":"YYYY-MM-DD","title":string,"description":string,"tag":string}}]}}

[입력]
{{input_json}}
""",
    "quest_generation": f"""{SERVICE_TONE}

너는 몽글마을 캐릭터 퀘스트 생성기다.
사용자 TODO 하나와 캐릭터 정보를 보고, 캐릭터가 마을에서 할 독립적인 작은 행동을 만든다.

규칙:
- JSON으로만 응답한다. 마크다운 코드블록은 쓰지 않는다.
- 출력 스키마: {{"quest": string}}
- 퀘스트는 사용자 TODO를 직접 따라 하거나 핵심 단어를 노출하면 안 된다.
- 캐릭터의 페르소나, 외형, 따뜻한 마을 생활을 반영한다.
- 짧고 귀엽지만 과하게 유치하지 않은 동사형 문장으로 쓴다.

[입력]
{{input_json}}
""",
    "feed_post_generation": f"""{SERVICE_TONE}

너는 몽글마을 캐릭터 피드 글 생성기다.
완료된 캐릭터 퀘스트를 바탕으로 캐릭터가 직접 SNS에 올린 짧은 글을 만든다.

규칙:
- JSON으로만 응답한다. 마크다운 코드블록은 쓰지 않는다.
- 출력 스키마: {{"post": string}}
- 게시글은 한국어 140자 이내다.
- 완료 퀘스트와 캐릭터 페르소나를 자연스럽게 반영한다.
- 사용자의 실제 TODO를 직접 언급하지 않는다.
- AI/시스템 안내문처럼 쓰지 않는다.

[입력]
{{input_json}}
""",
    "character_message_generation": f"""{SERVICE_TONE}

너는 몽글마을 캐릭터 말투/알림 메시지 생성기다.
이벤트와 메시지 목표를 보고 캐릭터가 사용자에게 건넬 짧은 말을 만든다.

규칙:
- JSON으로만 응답한다. 마크다운 코드블록은 쓰지 않는다.
- 출력 스키마: {{"message": string}}
- 메시지는 한국어 120자 이내를 권장한다.
- 이벤트와 관련 있어야 한다.
- event가 "피드 댓글 답글"이면 user_comment의 내용과 질문에 직접 관련된 답글을 만든다.
- 답글은 feed_post와 completed_quest 맥락을 자연스럽게 반영하되 사용자 TODO나 개인정보를 드러내지 않는다.
- 사용자를 재촉하거나 비난하지 않는다.
- 캐릭터 이름은 자연스러울 때만 사용한다.

[입력]
{{input_json}}
""",
}

def build_prompt(task: str, input_json: Dict[str, Any]) -> str:
    input_text = json.dumps(input_json, ensure_ascii=False, indent=2)
    return PROMPT_TEMPLATES[task].replace("{input_json}", input_text)


## 7. Dataset Loader

In [9]:

DATASET_PATHS = {
    "characters": FIXTURE_DIR / "characters.sample.json",
    "todo_decomposition": GENERATED_DIR / "todo_decomposition_samples.json",
    "planner_chatbot": GENERATED_DIR / "planner_samples.json",
    "quest_generation": GENERATED_DIR / "quest_samples.json",
    "feed_post_generation": GENERATED_DIR / "feed_samples.json",
    "character_message_generation": GENERATED_DIR / "character_message_samples.json",
}

def read_json(path: Path, default: Any) -> Any:
    if not path.exists():
        print(f"파일 없음, 최소 샘플 사용: {path.relative_to(PROJECT_ROOT)}")
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

MINIMAL_CHARACTER = {
    "character_id": "CHAR-SAMPLE-001",
    "name": "최벨리",
    "appearance": {"animal_type": "곰", "main_color": "핑크", "body_shape": "둥근 체형"},
    "persona": "덩치는 크지만 세상에서 제일 따뜻한 곰. 먹을 걸 챙겨두고 아낌없이 나눠주는 든든한 친구.",
    "personality_keywords": ["따뜻한", "든든한", "다정한", "포근한"],
}

MINIMAL_DATASETS = {
    "todo_decomposition": [
        {
            "todo_sample_id": "TODO-MIN-001",
            "user_message": "오늘 운동하고 프로젝트 기획서 쓰고 장보기 해야 해.",
            "todos": [
                {"title": "운동하기", "tag": "운동", "date": "오늘", "time": None},
                {"title": "프로젝트 기획서 작성하기", "tag": "프로젝트", "date": "오늘", "time": None},
                {"title": "장보기", "tag": "생활", "date": "오늘", "time": None},
            ],
        }
    ],
    "planner_chatbot": [
        {
            "planner_sample_id": "PLAN-MIN-001",
            "user_message": "3일 후 시험인데 아직 범위를 다 못 봤어. 일정 나눠줘.",
            "current_date": "2026-05-20",
            "output": {"response_type": "clarifying_question", "message": "하루에 몇 시간 정도 공부할 수 있어?", "missing_fields": ["available_hours_per_day"]},
        }
    ],
    "quest_generation": [
        {
            "quest_id": "QUEST-MIN-001",
            "user_todo": "깃 커밋하기",
            "character": MINIMAL_CHARACTER,
            "quest": "따뜻한 간식 바구니 나눠주기",
        }
    ],
    "feed_post_generation": [
        {
            "feed_sample_id": "FEED-MIN-001",
            "character": MINIMAL_CHARACTER,
            "completed_quest": "따뜻한 간식 바구니 나눠주기",
            "post": "오늘은 간식 바구니를 채워 나눠줬어. 누군가 웃는 걸 보니까 마음까지 포근해졌어.",
        }
    ],
    "character_message_generation": [
        {
            "message_sample_id": "MSG-MIN-001",
            "character": MINIMAL_CHARACTER,
            "event": "TODO 완료",
            "message_goal": "사용자를 부드럽게 칭찬하고 다음 행동을 압박하지 않는다.",
            "message": "오늘도 하나 해냈구나. 최벨리가 따뜻한 간식 하나 챙겨두고 기다리고 있었어.",
        }
    ],
}

def normalize_sample(task: str, sample: Dict[str, Any], characters: List[Dict[str, Any]]) -> Tuple[str, Dict[str, Any], Dict[str, Any]]:
    character = characters[0] if characters else MINIMAL_CHARACTER

    if task == "todo_decomposition":
        sample_id = sample.get("sample_id") or sample.get("todo_sample_id", "TODO-UNKNOWN")
        input_json = {"sample_id": sample_id, "user_message": sample.get("user_message", "")}
        expected = {"todos": sample.get("todos", []), "expected_todo_count": len(sample.get("todos", []))}
    elif task == "planner_chatbot":
        sample_id = sample.get("sample_id") or sample.get("planner_sample_id", "PLAN-UNKNOWN")
        input_json = {"sample_id": sample_id, "user_message": sample.get("user_message", ""), "current_date": sample.get("current_date")}
        expected = sample.get("output", {})
    elif task == "quest_generation":
        sample_id = sample.get("sample_id") or sample.get("quest_id", "QUEST-UNKNOWN")
        sample_character = sample.get("character") or {
            "name": sample.get("character_name", character.get("name")),
            "persona": character.get("persona") or character.get("personaPrompt"),
            "appearance": {
                "animal_type": sample.get("character_type", "인형"),
                "keywords": sample.get("appearance_keywords", []),
            },
            "personality_keywords": sample.get("personality", []),
        }
        input_json = {"sample_id": sample_id, "user_todo": sample.get("user_todo", "깃 커밋하기"), "character": sample_character}
        expected = {"quest": sample.get("quest")}
    elif task == "feed_post_generation":
        sample_id = sample.get("sample_id") or sample.get("feed_sample_id", "FEED-UNKNOWN")
        sample_character = sample.get("character") or {
            "name": sample.get("character_name", character.get("name")),
            "persona": character.get("persona") or character.get("personaPrompt"),
            "personality_keywords": sample.get("personality", []),
        }
        input_json = {"sample_id": sample_id, "character": sample_character, "completed_quest": sample.get("completed_quest")}
        expected = {"post": sample.get("post")}
    else:
        sample_id = sample.get("sample_id") or sample.get("message_sample_id", "MSG-UNKNOWN")
        # 답글 생성 샘플은 feed_post, user_comment 같은 맥락 필드를 함께 전달합니다.
        input_json = {
            "sample_id": sample_id,
            "character": sample.get("character", character),
            "event": sample.get("event", "TODO 완료"),
            "message_goal": sample.get("message_goal", "사용자를 부드럽게 응원한다."),
        }
        for optional_key in ["feed_post", "completed_quest", "user_comment", "comment_intent", "reply_constraints"]:
            if optional_key in sample:
                input_json[optional_key] = sample[optional_key]
        expected = {"message": sample.get("message")}

    return sample_id, input_json, expected

def load_datasets() -> Dict[str, List[Dict[str, Any]]]:
    characters_data = read_json(DATASET_PATHS["characters"], [MINIMAL_CHARACTER])
    characters = characters_data if isinstance(characters_data, list) else [characters_data]

    datasets = {}
    for task in PROMPT_TEMPLATES:
        raw_samples = read_json(DATASET_PATHS[task], MINIMAL_DATASETS[task])
        normalized = []
        for sample in raw_samples:
            sample_id, input_json, expected = normalize_sample(task, sample, characters)
            normalized.append({"sample_id": sample_id, "input": input_json, "expected": expected})
        datasets[task] = normalized[:MAX_SAMPLES_PER_TASK] if MAX_SAMPLES_PER_TASK else normalized
    return datasets

DATASETS = load_datasets()
{task: len(samples) for task, samples in DATASETS.items()}


{'todo_decomposition': 20,
 'planner_chatbot': 20,
 'quest_generation': 20,
 'feed_post_generation': 20,
 'character_message_generation': 10}

## 7-1. Runtime Dependency Check

In [10]:

def check_runtime_dependencies() -> None:
    """실제 모델 실행 전에 핵심 라이브러리 import 문제를 먼저 보여줍니다."""
    checks = [
        ("torch", "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"),
        ("transformers", "import transformers; print('transformers', transformers.__version__)"),
        ("AutoTokenizer", "from transformers import AutoTokenizer; print('AutoTokenizer import OK')"),
        ("AutoModelForCausalLM", "from transformers import AutoModelForCausalLM; print('AutoModelForCausalLM import OK')"),
        ("accelerate", "import accelerate; print('accelerate', accelerate.__version__)"),
    ]
    for name, stmt in checks:
        try:
            exec(stmt, globals(), globals())
        except Exception as exc:
            print(f"[{name} import failed] {type(exc).__name__}: {exc}")
            print("실제 모델 평가 전에 이 의존성 문제를 먼저 해결해야 합니다.")
            print("\n전체 traceback:")
            print(traceback.format_exc())
            break

check_runtime_dependencies()


torch 2.13.0+cu130 cuda True
transformers 5.9.0
AutoTokenizer import OK
AutoModelForCausalLM import OK
accelerate 1.14.0


### RunPod dependency repair

`LlamaConfig` import 에러가 나면 `transformers`/`torch`/부가 패키지 조합이 맞지 않는 경우가 많습니다. 아래 셀은 필요할 때만 주석을 해제해서 실행하세요. 실행 후에는 커널을 재시작해야 합니다.

In [11]:
# RunPod dependency repair - 필요할 때만 주석을 해제하세요.
# !pip install -U --upgrade-strategy eager "transformers>=4.51.0" "accelerate>=0.34.0" "safetensors" "sentencepiece" "protobuf"
# !pip install -U --upgrade-strategy eager "torch"
# 설치 후 Kernel > Restart Kernel and Run All 을 실행하세요.


## 8. Provider Adapters

In [12]:

class BaseProvider:
    def generate(self, prompt: str, generation_config: Dict[str, Any]) -> str:
        raise NotImplementedError

# class MockProvider(BaseProvider):
#     """GPU/API 없이 평가 파이프라인을 확인하기 위한 provider입니다."""
#     # 얘는 무시 하세요

#     def __init__(self, task: str):
#         self.task = task

#     def generate(self, prompt: str, generation_config: Dict[str, Any]) -> str:
#         if self.task == "todo_decomposition":
#             return json.dumps({"todos": [{"title": "운동하기", "tag": "운동", "date": "오늘", "time": None}]}, ensure_ascii=False)
#         if self.task == "planner_chatbot":
#             return json.dumps({"response_type": "clarifying_question", "message": "하루에 몇 시간 정도 쓸 수 있어?", "missing_fields": ["available_hours_per_day"]}, ensure_ascii=False)
#         if self.task == "quest_generation":
#             return json.dumps({"quest": "따뜻한 간식 바구니 나눠주기"}, ensure_ascii=False)
#         if self.task == "feed_post_generation":
#             return json.dumps({"post": "오늘은 간식 바구니를 나눠줬어. 작은 웃음들이 모여서 마음이 포근해졌어."}, ensure_ascii=False)
#         return json.dumps({"message": "오늘도 하나 해냈구나. 천천히 쉬어가도 괜찮아."}, ensure_ascii=False)

class TransformersProvider(BaseProvider):
    _cache: Dict[Tuple[str, str], Tuple[Any, Any]] = {}

    def __init__(self, model_id: str, hf_token: str = "", tokenizer_kwargs: Optional[Dict[str, Any]] = None):
        self.model_id = model_id
        self.hf_token = hf_token or None
        self.tokenizer_kwargs = tokenizer_kwargs or {}

    def _load(self):
        cache_key = (self.model_id, json.dumps(self.tokenizer_kwargs, sort_keys=True))
        if cache_key in self._cache:
            return self._cache[cache_key]

        from transformers import AutoModelForCausalLM, AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(
            self.model_id,
            token=self.hf_token,
            trust_remote_code=True,
            **self.tokenizer_kwargs,
        )
        import torch

        model_kwargs = {
            "token": self.hf_token,
            "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else "auto",
            "trust_remote_code": True,
        }
        if torch.cuda.is_available():
            model_kwargs["device_map"] = {"": 0} if FORCE_SINGLE_GPU else "auto"
        else:
            model_kwargs["device_map"] = "auto"

        try:
            model = AutoModelForCausalLM.from_pretrained(self.model_id, attn_implementation="sdpa", **model_kwargs)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(self.model_id, **model_kwargs)
        model.eval()

        if PRINT_MODEL_PLACEMENT:
            first_param = next(model.parameters())
            print(f"[{self.model_id}] device={first_param.device}, dtype={first_param.dtype}")
            if hasattr(model, "hf_device_map"):
                print("device_map:", model.hf_device_map)

        self._cache[cache_key] = (tokenizer, model)
        return tokenizer, model

    def generate(self, prompt: str, generation_config: Dict[str, Any]) -> str:
        tokenizer, model = self._load()
        messages = [{"role": "user", "content": prompt}]

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            text = prompt

        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        generate_kwargs = {
            "max_new_tokens": generation_config.get("max_new_tokens", 1024),
            "temperature": generation_config.get("temperature", 0.3),
            "top_p": generation_config.get("top_p", 0.9),
            "do_sample": generation_config.get("do_sample", True),
            "repetition_penalty": generation_config.get("repetition_penalty", 1.0),
            "pad_token_id": tokenizer.eos_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        import torch
        with torch.inference_mode():
            outputs = model.generate(**inputs, **generate_kwargs)
        new_tokens = outputs[0][input_len:]
        self.last_generated_tokens = int(new_tokens.shape[-1])
        return tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()

class OpenAICompatibleProvider(BaseProvider):
    def __init__(self, model_id: str, base_url: str, api_key: str):
        from openai import OpenAI
        self.model_id = model_id
        self.client = OpenAI(base_url=base_url, api_key=api_key)

    def generate(self, prompt: str, generation_config: Dict[str, Any]) -> str:
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=generation_config.get("temperature", 0.3),
            top_p=generation_config.get("top_p", 0.9),
            max_tokens=generation_config.get("max_new_tokens", 1024),
        )
        return response.choices[0].message.content.strip()

def should_skip_model(config: Dict[str, Any]) -> Optional[str]:
    model_id = config["model_id"]
    if config.get("provider") == "transformers" and not ENV["HF_TOKEN"] and model_id.startswith(GATED_MODEL_PREFIXES):
        return "HF_TOKEN이 없어 gated 가능성이 있는 모델을 건너뜀"
    if config.get("provider") == "openai_compatible" and not ENV["VLLM_BASE_URL"]:
        return "VLLM_BASE_URL이 없어 건너뜀"
    return None

def make_provider(model_key: str, config: Dict[str, Any], task: str) -> BaseProvider:
    if RUN_MOCK_ONLY:
        return MockProvider(task)

    provider = config["provider"]
    if provider == "transformers":
        return TransformersProvider(config["model_id"], ENV["HF_TOKEN"], config.get("tokenizer_kwargs"))
    if provider == "openai_compatible":
        return OpenAICompatibleProvider(config["model_id"], ENV["VLLM_BASE_URL"], ENV["VLLM_API_KEY"])
    if provider == "mock":
        return MockProvider(task)
    raise ValueError(f"지원하지 않는 provider: {provider}")


## 9. JSON Parser

In [13]:

def extract_json_text(text: str) -> str:
    text = (text or "").strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    if text.startswith("{") or text.startswith("["):
        return text

    # 모델이 앞뒤 설명을 붙인 경우 첫 JSON 객체/배열만 최대한 추출합니다.
    starts = [pos for pos in [text.find("{"), text.find("[")] if pos != -1]
    if not starts:
        return text
    start = min(starts)
    end_obj = text.rfind("}")
    end_arr = text.rfind("]")
    end = max(end_obj, end_arr)
    return text[start:end + 1] if end >= start else text[start:]

def parse_json_output(raw_output: str) -> Tuple[Optional[Any], Optional[str]]:
    try:
        return json.loads(extract_json_text(raw_output)), None
    except Exception as exc:
        return None, str(exc)


## 10. Auto Evaluators

In [14]:

FORBIDDEN_PATTERNS = [
    "죽", "살해", "폭력", "혐오", "성적", "협박", "멍청", "한심", "게으르", "당장", "안 하면", "실패자",
]
PRESSURE_PATTERNS = ["당장", "빨리", "안 하면", "해야만", "게으르", "한심", "실패"]
KOREAN_RE = re.compile(r"[가-힣]")
DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

TASK_REQUIRED_FIELDS = {
    "todo_decomposition": ["todos"],
    "planner_chatbot": ["response_type"],
    "quest_generation": ["quest"],
    "feed_post_generation": ["post"],
    "character_message_generation": ["message"],
}

TEXT_FIELD_BY_TASK = {
    "quest_generation": "quest",
    "feed_post_generation": "post",
    "character_message_generation": "message",
}

LENGTH_LIMITS = {
    "planner_chatbot": 1500,
    "quest_generation": 40,
    "feed_post_generation": 140,
    "character_message_generation": 120,
}

def has_korean(value: Any) -> bool:
    return bool(KOREAN_RE.search(json.dumps(value, ensure_ascii=False)))

def contains_forbidden(value: Any) -> bool:
    text = json.dumps(value, ensure_ascii=False)
    return any(pattern in text for pattern in FORBIDDEN_PATTERNS)

def text_length_for_task(task: str, parsed: Any) -> int:
    if not isinstance(parsed, dict):
        return len(str(parsed or ""))
    field = TEXT_FIELD_BY_TASK.get(task)
    if field:
        return len(str(parsed.get(field, "")))
    return len(json.dumps(parsed, ensure_ascii=False))

def duplicate_rate(items: List[str]) -> float:
    cleaned = [item.strip() for item in items if item and item.strip()]
    if not cleaned:
        return 0.0
    return 1 - (len(set(cleaned)) / len(cleaned))

def tokenize_ko(text: str) -> List[str]:
    return re.findall(r"[가-힣A-Za-z0-9]+", text or "")

def distinct_n(texts: List[str], n: int) -> float:
    grams = []
    for text in texts:
        tokens = tokenize_ko(text)
        grams.extend(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))
    return len(set(grams)) / len(grams) if grams else 0.0

def overlap_rate(a: str, b: str) -> float:
    a_tokens = set(tokenize_ko(a))
    b_tokens = set(tokenize_ko(b))
    if not a_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens)

def speed_score_from_latency(task: str, latency_sec: Optional[float]) -> Optional[float]:
    if latency_sec is None:
        return None
    threshold = SPEED_THRESHOLDS.get(task, DEFAULT_SPEED_THRESHOLD)
    fast = threshold["fast"]
    slow = threshold["slow"]
    if latency_sec <= fast:
        return 5.0
    if latency_sec >= slow:
        return 1.0
    # fast 이하는 5점, slow 이상은 1점으로 두고 선형 보간합니다.
    score = 1 + 4 * (slow - latency_sec) / (slow - fast)
    return round(max(1.0, min(5.0, score)), 2)

def common_eval(task: str, parsed: Any, parse_error: Optional[str], raw_output: str, latency_sec: float, error: Optional[str]) -> Dict[str, Any]:
    schema_valid = isinstance(parsed, dict) and all(field in parsed for field in TASK_REQUIRED_FIELDS[task])
    length_limit = LENGTH_LIMITS.get(task)
    length = text_length_for_task(task, parsed if parsed is not None else raw_output)
    return {
        "json_valid": parse_error is None,
        "schema_valid": schema_valid,
        "korean_output": has_korean(parsed if parsed is not None else raw_output),
        "length_valid": True if length_limit is None else length <= length_limit,
        "output_length": length,
        "forbidden_detected": contains_forbidden(parsed if parsed is not None else raw_output),
        "error": error is not None,
        "latency_sec": latency_sec,
        "speed_score": speed_score_from_latency(task, latency_sec),
    }

def eval_todo(parsed: Any, expected: Dict[str, Any]) -> Dict[str, Any]:
    todos = parsed.get("todos", []) if isinstance(parsed, dict) else []
    titles = [todo.get("title", "") for todo in todos if isinstance(todo, dict)]
    expected_todos = expected.get("todos", [])
    expected_count = expected.get("expected_todo_count", len(expected_todos))
    expected_titles = [todo.get("title", "") for todo in expected_todos]
    expected_tags = [todo.get("tag") for todo in expected_todos]
    tags = [todo.get("tag") for todo in todos if isinstance(todo, dict)]

    return {
        "todo_count": len(todos),
        "todo_count_accuracy": 1.0 if len(todos) == expected_count else max(0.0, 1 - abs(len(todos) - expected_count) / max(expected_count, 1)),
        "empty_todo_rate": sum(1 for title in titles if not title.strip()) / max(len(titles), 1),
        "duplicate_todo_rate": duplicate_rate(titles),
        "tag_presence_rate": sum(1 for tag in tags if tag) / max(len(todos), 1),
        "date_field_rate": sum(1 for todo in todos if isinstance(todo, dict) and "date" in todo) / max(len(todos), 1),
        "time_field_rate": sum(1 for todo in todos if isinstance(todo, dict) and "time" in todo) / max(len(todos), 1),
        "tag_accuracy_hint": sum(1 for tag in tags if tag in expected_tags) / max(len(tags), 1),
        "task_coverage_hint": sum(1 for exp in expected_titles if any(overlap_rate(exp, title) >= 0.4 for title in titles)) / max(len(expected_titles), 1),
    }

def eval_planner(parsed: Any, expected: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(parsed, dict):
        return {"response_type_valid": False, "response_type_accuracy": 0.0, "plan_schema_valid": False}
    response_type = parsed.get("response_type")
    expected_type = expected.get("response_type")
    todos = parsed.get("todos", [])
    events = parsed.get("calendar_events", [])
    is_question = response_type == "clarifying_question"
    is_plan = response_type == "plan"
    dates = [item.get("date") for item in todos + events if isinstance(item, dict)] if is_plan else []
    date_valid = all(isinstance(date, str) and bool(DATE_RE.match(date)) for date in dates)
    return {
        "response_type_valid": response_type in {"clarifying_question", "plan"},
        "response_type_accuracy": 1.0 if expected_type and response_type == expected_type else 0.0,
        "clarifying_schema_valid": is_question and isinstance(parsed.get("message"), str) and isinstance(parsed.get("missing_fields"), list),
        "plan_schema_valid": is_plan and isinstance(todos, list) and isinstance(events, list),
        "date_valid": True if not dates else date_valid,
        "todo_convertibility": sum(1 for item in todos if isinstance(item, dict) and item.get("date") and item.get("title") and item.get("tag")) / max(len(todos), 1),
        "calendar_convertibility": sum(1 for item in events if isinstance(item, dict) and item.get("date") and item.get("title") and item.get("description") and item.get("tag")) / max(len(events), 1),
        "tag_coverage": sum(1 for item in todos + events if isinstance(item, dict) and item.get("tag")) / max(len(todos) + len(events), 1),
    }

def eval_quest(parsed: Any, input_json: Dict[str, Any]) -> Dict[str, Any]:
    quest = parsed.get("quest", "") if isinstance(parsed, dict) else ""
    user_todo = input_json.get("user_todo", "")
    character_text = json.dumps(input_json.get("character", {}), ensure_ascii=False)
    return {
        "todo_direct_overlap_rate": overlap_rate(user_todo, quest),
        "todo_direct_related_violation": overlap_rate(user_todo, quest) >= 0.45,
        "persona_keyword_hit_rate": overlap_rate(quest, character_text),
        "worldview_term_hit": any(word in quest for word in ["마을", "리본", "찻잔", "바구니", "담요", "창가", "꽃", "구름", "간식", "정리"]),
        "distinct_1": distinct_n([quest], 1),
        "distinct_2": distinct_n([quest], 2),
    }

def eval_feed(parsed: Any, input_json: Dict[str, Any]) -> Dict[str, Any]:
    post = parsed.get("post", "") if isinstance(parsed, dict) else ""
    user_todo = input_json.get("user_todo", "") or ""
    completed_quest = input_json.get("completed_quest", "") or ""
    return {
        "quest_mention_hint": overlap_rate(completed_quest, post),
        "user_todo_leak_rate": overlap_rate(user_todo, post) if user_todo else 0.0,
        "repetitive_phrase_hint": duplicate_rate(tokenize_ko(post)),
        "sns_post_valid": bool(post) and len(post) <= 140 and "입니다" not in post[:20],
        "distinct_1": distinct_n([post], 1),
        "distinct_2": distinct_n([post], 2),
    }

def eval_message(parsed: Any, input_json: Dict[str, Any]) -> Dict[str, Any]:
    message = parsed.get("message", "") if isinstance(parsed, dict) else ""
    character_name = input_json.get("character", {}).get("name", "") if isinstance(input_json.get("character"), dict) else ""
    event = input_json.get("event", "")
    user_comment = input_json.get("user_comment", "")
    feed_post = input_json.get("feed_post", "")
    return {
        "event_relevance_hint": overlap_rate(event, message),
        "comment_reply_relevance_hint": overlap_rate(user_comment, message) if user_comment else None,
        "feed_context_relevance_hint": overlap_rate(feed_post, message) if feed_post else None,
        "pressure_phrase_detected": any(pattern in message for pattern in PRESSURE_PATTERNS),
        "comfort_phrase_hit": any(word in message for word in ["괜찮", "잘했", "해냈", "쉬어", "고생", "포근", "응원", "그랬구나"]),
        "character_name_used": bool(character_name and character_name in message),
    }

def auto_evaluate(task: str, parsed: Any, parse_error: Optional[str], raw_output: str, input_json: Dict[str, Any], expected: Dict[str, Any], latency_sec: float, error: Optional[str]) -> Dict[str, Any]:
    result = common_eval(task, parsed, parse_error, raw_output, latency_sec, error)
    if parsed is None:
        return result
    if task == "todo_decomposition":
        result.update(eval_todo(parsed, expected))
    elif task == "planner_chatbot":
        result.update(eval_planner(parsed, expected))
    elif task == "quest_generation":
        result.update(eval_quest(parsed, input_json))
    elif task == "feed_post_generation":
        result.update(eval_feed(parsed, input_json))
    elif task == "character_message_generation":
        result.update(eval_message(parsed, input_json))
    return result

def metric_score(value: Any, higher_is_better: bool = True) -> float:
    if isinstance(value, bool):
        score = 1.0 if value else 0.0
    elif isinstance(value, (int, float)):
        score = max(0.0, min(1.0, float(value)))
    else:
        score = 0.0
    return score if higher_is_better else 1.0 - score

def auto_score_5pt(task: str, auto_eval: Dict[str, Any]) -> float:
    base = [
        metric_score(auto_eval.get("json_valid")),
        metric_score(auto_eval.get("schema_valid")),
        metric_score(auto_eval.get("korean_output")),
        metric_score(auto_eval.get("length_valid")),
        metric_score(auto_eval.get("forbidden_detected"), higher_is_better=False),
        metric_score(auto_eval.get("error"), higher_is_better=False),
    ]
    task_metrics = []
    if task == "todo_decomposition":
        task_metrics = [
            auto_eval.get("todo_count_accuracy", 0),
            1 - auto_eval.get("duplicate_todo_rate", 1),
            auto_eval.get("tag_presence_rate", 0),
            auto_eval.get("task_coverage_hint", 0),
        ]
    elif task == "planner_chatbot":
        task_metrics = [
            auto_eval.get("response_type_accuracy", 0),
            auto_eval.get("todo_convertibility", 0),
            auto_eval.get("calendar_convertibility", 0),
            auto_eval.get("tag_coverage", 0),
        ]
    elif task == "quest_generation":
        task_metrics = [
            1 - float(auto_eval.get("todo_direct_related_violation", True)),
            auto_eval.get("persona_keyword_hit_rate", 0),
            float(auto_eval.get("worldview_term_hit", False)),
        ]
    elif task == "feed_post_generation":
        task_metrics = [
            auto_eval.get("quest_mention_hint", 0),
            1 - auto_eval.get("user_todo_leak_rate", 0),
            float(auto_eval.get("sns_post_valid", False)),
        ]
    elif task == "character_message_generation":
        task_metrics = [
            1 - float(auto_eval.get("pressure_phrase_detected", True)),
            float(auto_eval.get("comfort_phrase_hit", False)),
            auto_eval.get("event_relevance_hint", 0),
        ]
    values = base + [metric_score(v) for v in task_metrics]
    # 모든 점수는 1~5점 만점으로 저장합니다.
    score = 1 + 4 * (sum(values) / max(len(values), 1))
    return round(max(1.0, min(5.0, score)), 2)


## 11. GPT-5.5 Judge Evaluator

In [15]:
JUDGE_PROMPT_TEMPLATE = """
너는 몽글마을 서비스의 LLM 응답 품질을 평가하는 심판 모델이다.

몽글마을은 사용자의 애착인형을 AI 캐릭터로 생성하고, TODO·일정·퀘스트·피드·토큰을 연결하는 감성형 생산성 서비스다.

[서비스 톤]
- 한국어 우선
- 따뜻하고 다정함
- 귀엽지만 과하게 유치하지 않음
- 사용자를 비난하거나 압박하지 않음
- 캐릭터가 마을 주민처럼 살아가는 느낌
- 사용자 TODO를 캐릭터 퀘스트/피드에 직접 노출하지 않음

[평가 대상 기능]
{task_name}

[입력]
{input_json}

[모델 응답]
{model_output}

각 항목을 1~5점으로 평가해라. 반드시 JSON으로만 응답해라.

{{
  "persona_reflection": {{"score": 1, "reason": ""}},
  "character_consistency": {{"score": 1, "reason": ""}},
  "tone_distinctiveness": {{"score": 1, "reason": ""}},
  "task_requirement_fit": {{"score": 1, "reason": ""}},
  "emotional_comfort": {{"score": 1, "reason": ""}},
  "format_following": {{"score": 1, "reason": ""}},
  "repetition_or_character_break": {{"score": 1, "reason": ""}},
  "overall": {{"score": 1, "reason": ""}}
}}
""".strip()

def judge_evaluate(record: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not JUDGE_CONFIG.get("enabled"):
        return None
    if not ENV["OPENAI_API_KEY"]:
        print("OPENAI_API_KEY가 없어 GPT-5.5 심판 평가를 건너뜁니다.")
        return None

    from openai import OpenAI
    client = OpenAI(api_key=ENV["OPENAI_API_KEY"])
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        task_name=record["task"],
        input_json=json.dumps(record["input"], ensure_ascii=False, indent=2),
        model_output=record["raw_output"],
    )

    base_result = {
        "model_key": record["model_key"],
        "task": record["task"],
        "sample_id": record["sample_id"],
        "judge_model": JUDGE_CONFIG["model_id"],
    }

    try:
        if JUDGE_CONFIG.get("api", "responses") == "responses":
            response = client.responses.create(
                model=JUDGE_CONFIG["model_id"],
                input=[{"role": "user", "content": prompt}],
                max_output_tokens=JUDGE_CONFIG["max_output_tokens"],
            )
            raw = getattr(response, "output_text", "") or ""
        else:
            response = client.chat.completions.create(
                model=JUDGE_CONFIG["model_id"],
                messages=[{"role": "user", "content": prompt}],
                max_completion_tokens=JUDGE_CONFIG["max_output_tokens"],
            )
            raw = response.choices[0].message.content or ""
    except Exception as exc:
        base_result["judge_error"] = repr(exc)
        print(f"[judge skipped] {record['model_key']}/{record['task']}/{record['sample_id']}: {exc}")
        return base_result

    parsed, parse_error = parse_json_output(raw)
    if parse_error:
        base_result.update({"judge_parse_error": parse_error, "judge_raw_output": raw})
        return base_result

    scores = {}
    reasons = {}
    for key, value in parsed.items():
        if isinstance(value, dict):
            scores[key] = value.get("score")
            reasons[key] = value.get("reason")
    base_result.update({"judge_scores": scores, "judge_reasons": reasons})
    return base_result

def normalized_judge_score(judge_record: Optional[Dict[str, Any]]) -> Optional[float]:
    if not judge_record or "judge_scores" not in judge_record:
        return None
    scores = [v for v in judge_record["judge_scores"].values() if isinstance(v, (int, float))]
    return round(sum(scores) / len(scores), 2) if scores else None


## 11-1. Model Preload / Warmup

실제 평가 전에 enabled 모델을 한 번만 미리 로드하고 짧게 warmup합니다. 이 셀을 먼저 실행하면 첫 샘플에 모델 다운로드/로드 시간이 섞이는 것을 줄일 수 있습니다. 커널을 재시작하거나 Provider 셀을 다시 실행하면 캐시는 초기화됩니다.

In [16]:

PRELOAD_MODELS = True
WARMUP_AFTER_PRELOAD = True
WARMUP_PROMPT = "JSON으로만 응답해. {\"message\":\"준비 완료\"}"

PRELOADED_PROVIDERS: Dict[str, BaseProvider] = {}

def preload_enabled_models() -> Dict[str, BaseProvider]:
    """enabled=True인 transformers 모델을 평가 전에 미리 로드합니다."""
    if RUN_MOCK_ONLY:
        print("RUN_MOCK_ONLY=True라 실제 모델 preload는 건너뜁니다.")
        return {}

    providers = {}
    enabled_models = {key: cfg for key, cfg in MODEL_CONFIGS.items() if cfg.get("enabled")}
    for model_key, model_config in enabled_models.items():
        if model_config.get("provider") != "transformers":
            continue
        skip_reason = should_skip_model(model_config)
        if skip_reason:
            print(f"[{model_key}] {skip_reason}")
            continue

        print(f"\npreload 시작: {model_key}")
        started = time.time()
        provider = make_provider(model_key, model_config, "todo_decomposition")
        try:
            # _load()가 모델/토크나이저를 TransformersProvider._cache에 저장합니다.
            provider._load()
            providers[model_key] = provider
            print(f"preload 완료: {model_key} ({time.time() - started:.2f}s)")

            if WARMUP_AFTER_PRELOAD:
                warmup_started = time.time()
                provider.generate(WARMUP_PROMPT, {"max_new_tokens": 16, "do_sample": False, "temperature": 0.0})
                print(f"warmup 완료: {model_key} ({time.time() - warmup_started:.2f}s)")
        except Exception as exc:
            print(f"preload 실패: {model_key}: {type(exc).__name__}: {exc}")
            print(traceback.format_exc())

    return providers

if PRELOAD_MODELS:
    PRELOADED_PROVIDERS = preload_enabled_models()
else:
    print("PRELOAD_MODELS=False라 preload를 건너뜁니다.")



preload 시작: midm-mini


config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/21.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

modeling_phi3.py:   0%|          | 0.00/54.3k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


preload 실패: phi-4-mini: ImportError: cannot import name 'LossKwargs' from 'transformers.utils' (/usr/local/lib/python3.11/dist-packages/transformers/utils/__init__.py)
Traceback (most recent call last):
  File "/tmp/ipykernel_2744/3418778934.py", line 28, in preload_enabled_models
    provider._load()
  File "/tmp/ipykernel_2744/3987151760.py", line 57, in _load
    model = AutoModelForCausalLM.from_pretrained(self.model_id, attn_implementation="sdpa", **model_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py", line 379, in from_pretrained
    model_class = get_class_from_dynamic_module(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/dynamic_module_utils.py", line 627, in get_class_from_dynamic_module
    return get_class_in_module(class_name, final_module, force_reload=f

## 12. Run Evaluation

In [17]:

def run_evaluation() -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    raw_records = []
    judge_records = []
    enabled_models = {key: cfg for key, cfg in MODEL_CONFIGS.items() if cfg.get("enabled")}

    if RUN_MOCK_ONLY:
        print("RUN_MOCK_ONLY=True: enabled 모델 키를 유지하되 MockProvider로 실행합니다.")

    for model_key, model_config in enabled_models.items():
        skip_reason = None if RUN_MOCK_ONLY else should_skip_model(model_config)
        if skip_reason:
            print(f"[{model_key}] {skip_reason}")
            continue

        print(f"\n모델 평가 시작: {model_key}")
        model_had_load_error = False
        for task, samples in DATASETS.items():
            if model_had_load_error:
                print(f"[{model_key}] 이전 단계에서 모델/라이브러리 로드 오류가 발생해 남은 task를 건너뜁니다.")
                break
            provider = None
            try:
                # preload 셀을 실행했다면 이미 로드된 provider/cache를 재사용합니다.
                provider = PRELOADED_PROVIDERS.get(model_key) if "PRELOADED_PROVIDERS" in globals() else None
                if provider is None:
                    provider = make_provider(model_key, model_config, task)
            except Exception as exc:
                print(f"[{model_key}/{task}] provider 생성 실패: {exc}")
                continue

            for sample in tqdm(samples, desc=f"{model_key}/{task}"):
                prompt = build_prompt(task, sample["input"])
                gen_config = generation_config_for(task)
                started = time.time()
                raw_output = ""
                error = None
                parsed = None
                parse_error = None

                try:
                    raw_output = provider.generate(prompt, gen_config)
                    parsed, parse_error = parse_json_output(raw_output)
                except Exception as exc:
                    error = f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}"
                    # transformers/torch import 또는 모델 로드 오류는 샘플별 문제가 아니라 환경/모델 단위 문제입니다.
                    # 같은 에러를 수백 줄 반복하지 않도록 현재 모델의 남은 평가를 건너뜁니다.
                    if isinstance(exc, (ModuleNotFoundError, ImportError)) or "Could not import module" in str(exc):
                        model_had_load_error = True

                latency_sec = round(time.time() - started, 4)
                generated_tokens = getattr(provider, "last_generated_tokens", None)
                auto_eval = auto_evaluate(task, parsed, parse_error, raw_output, sample["input"], sample["expected"], latency_sec, error)
                auto_eval["generated_tokens"] = generated_tokens
                auto_eval["tokens_per_sec"] = round(generated_tokens / latency_sec, 3) if generated_tokens and latency_sec > 0 else None
                auto_eval["auto_score"] = auto_score_5pt(task, auto_eval)

                record = {
                    "run_id": RUN_ID,
                    "model_key": model_key,
                    "model_id": model_config["model_id"],
                    "task": task,
                    "sample_id": sample["sample_id"],
                    "input": sample["input"],
                    "expected": sample["expected"],
                    "prompt": prompt,
                    "raw_output": raw_output,
                    "parsed_output": parsed,
                    "parse_error": parse_error,
                    "auto_eval": auto_eval,
                    "latency_sec": latency_sec,
                    "error": error,
                }
                raw_records.append(record)

                judge_record = judge_evaluate(record)
                if judge_record:
                    judge_records.append(judge_record)

                if model_had_load_error:
                    print(f"[{model_key}] 모델 로드/import 오류로 중단: {error}")
                    break

        if CLEAR_CUDA_AFTER_MODEL and not RUN_MOCK_ONLY:
            gc.collect()
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

    return raw_records, judge_records

RAW_RECORDS, JUDGE_RECORDS = run_evaluation()
print(f"raw records: {len(RAW_RECORDS)}")
print(f"judge records: {len(JUDGE_RECORDS)}")



모델 평가 시작: midm-mini


midm-mini/todo_decomposition:   0%|          | 0/20 [00:00<?, ?it/s]

midm-mini/planner_chatbot:   0%|          | 0/20 [00:00<?, ?it/s]

midm-mini/quest_generation:   0%|          | 0/20 [00:00<?, ?it/s]

midm-mini/feed_post_generation:   0%|          | 0/20 [00:00<?, ?it/s]

...

gemma-3-4b-it/feed_post_generation:   0%|          | 0/20 [00:00<?, ?it/s]

gemma-3-4b-it/character_message_generation:   0%|          | 0/10 [00:00<?, ?it/s]


모델 평가 시작: phi-4-mini


phi-4-mini/todo_decomposition:   0%|          | 0/20 [00:00<?, ?it/s]

[phi-4-mini] 모델 로드/import 오류로 중단: ImportError: cannot import name 'LossKwargs' from 'transformers.utils' (/usr/local/lib/python3.11/dist-packages/transformers/utils/__init__.py)
Traceback (most recent call last):
  File "/tmp/ipykernel_2744/3067048154.py", line 41, in run_evaluation
    raw_output = provider.generate(prompt, gen_config)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2744/3987151760.py", line 72, in generate
    tokenizer, model = self._load()
                       ^^^^^^^^^^^^
  File "/tmp/ipykernel_2744/3987151760.py", line 57, in _load
    model = AutoModelForCausalLM.from_pretrained(self.model_id, attn_implementation="sdpa", **model_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py", line 379, in from_pretrained
    model_class = get_class_from_dynamic_module(
                  ^^^^

## 13. Save Results

In [18]:

RAW_OUTPUTS_PATH = EVAL_DIR / "raw_outputs.jsonl"
AUTO_SCORES_PATH = EVAL_DIR / "auto_scores.json"
JUDGE_SCORES_PATH = EVAL_DIR / "judge_scores.jsonl"
FINAL_SUMMARY_PATH = EVAL_DIR / "final_summary.csv"
FAILURE_CASES_PATH = EVAL_DIR / "failure_cases.jsonl"

def write_jsonl(path: Path, records: List[Dict[str, Any]], append: bool = False) -> None:
    mode = "a" if append else "w"
    with path.open(mode, encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

def collect_failure_cases(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    failures = []
    for record in records:
        auto = record.get("auto_eval", {})
        failed = (
            record.get("error")
            or record.get("parse_error")
            or not auto.get("schema_valid", False)
            or auto.get("forbidden_detected", False)
            or not auto.get("length_valid", True)
            or auto.get("auto_score", 5) < 3.0
        )
        if failed:
            failures.append({
                "run_id": record["run_id"],
                "model_key": record["model_key"],
                "task": record["task"],
                "sample_id": record["sample_id"],
                "raw_output": record["raw_output"],
                "parse_error": record.get("parse_error"),
                "error": record.get("error"),
                "auto_eval": auto,
            })
    return failures

def save_results(raw_records: List[Dict[str, Any]], judge_records: List[Dict[str, Any]]) -> None:
    write_jsonl(RAW_OUTPUTS_PATH, raw_records, append=APPEND_OUTPUTS)
    write_jsonl(JUDGE_SCORES_PATH, judge_records, append=APPEND_OUTPUTS)
    failures = collect_failure_cases(raw_records)
    write_jsonl(FAILURE_CASES_PATH, failures, append=APPEND_OUTPUTS)

    auto_scores = {
        "run_id": RUN_ID,
        "created_at": datetime.now().isoformat(),
        "records": [
            {
                "model_key": r["model_key"],
                "task": r["task"],
                "sample_id": r["sample_id"],
                "auto_eval": r["auto_eval"],
            }
            for r in raw_records
        ],
    }
    AUTO_SCORES_PATH.write_text(json.dumps(auto_scores, ensure_ascii=False, indent=2), encoding="utf-8")

save_results(RAW_RECORDS, JUDGE_RECORDS)
print("저장 완료:")
for path in [RAW_OUTPUTS_PATH, AUTO_SCORES_PATH, JUDGE_SCORES_PATH, FINAL_SUMMARY_PATH, FAILURE_CASES_PATH]:
    print("-", path.relative_to(PROJECT_ROOT))


저장 완료:
- data/evaluation/raw_outputs.jsonl
- data/evaluation/auto_scores.json
- data/evaluation/judge_scores.jsonl
- data/evaluation/final_summary.csv
- data/evaluation/failure_cases.jsonl


## 14. Summary Tables

In [19]:

def build_summary(raw_records: List[Dict[str, Any]], judge_records: List[Dict[str, Any]]) -> pd.DataFrame:
    judge_map = {
        (j.get("model_key"), j.get("task"), j.get("sample_id")): normalized_judge_score(j)
        for j in judge_records
    }

    rows = []
    for record in raw_records:
        auto = record["auto_eval"]
        judge_score = judge_map.get((record["model_key"], record["task"], record["sample_id"]))
        final_score = None
        if judge_score is not None:
            final_score = round(auto["auto_score"] * 0.45 + judge_score * 0.55, 2)
        rows.append({
            "run_id": record["run_id"],
            "model_key": record["model_key"],
            "task": record["task"],
            "sample_id": record["sample_id"],
            "auto_score": auto.get("auto_score"),
            "json_valid": auto.get("json_valid"),
            "schema_valid": auto.get("schema_valid"),
            "korean_output": auto.get("korean_output"),
            "length_valid": auto.get("length_valid"),
            "forbidden_detected": auto.get("forbidden_detected"),
            "latency_sec": auto.get("latency_sec"),
            "generated_tokens": auto.get("generated_tokens"),
            "tokens_per_sec": auto.get("tokens_per_sec"),
            "speed_score": auto.get("speed_score"),
            "judge_score": judge_score,
            "final_score": final_score,
        })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    grouped = df.groupby(["model_key", "task"], dropna=False).agg(
        samples=("sample_id", "count"),
        auto_score=("auto_score", "mean"),
        json_parse_success_rate=("json_valid", "mean"),
        schema_success_rate=("schema_valid", "mean"),
        korean_output_rate=("korean_output", "mean"),
        length_success_rate=("length_valid", "mean"),
        forbidden_count=("forbidden_detected", "sum"),
        avg_latency_sec=("latency_sec", "mean"),
        total_latency_sec=("latency_sec", "sum"),
        avg_generated_tokens=("generated_tokens", "mean"),
        avg_tokens_per_sec=("tokens_per_sec", "mean"),
        speed_score=("speed_score", "mean"),
        judge_score=("judge_score", "mean"),
        final_score=("final_score", "mean"),
    ).reset_index()
    grouped["samples_per_min"] = grouped["samples"] / (grouped["total_latency_sec"] / 60).replace(0, pd.NA)

    numeric_cols = [
        "auto_score", "json_parse_success_rate", "schema_success_rate", "korean_output_rate",
        "length_success_rate", "avg_latency_sec", "total_latency_sec",
        "avg_generated_tokens", "avg_tokens_per_sec",
        "speed_score", "samples_per_min",
        "judge_score", "final_score",
    ]
    for col in numeric_cols:
        grouped[col] = pd.to_numeric(grouped[col], errors="coerce").round(3)
    return grouped

SUMMARY_DF = build_summary(RAW_RECORDS, JUDGE_RECORDS)
SUMMARY_DF.to_csv(FINAL_SUMMARY_PATH, index=False, encoding="utf-8-sig")
SUMMARY_DF


,model_key,task,samples,auto_score,json_parse_success_rate,schema_success_rate,korean_output_rate,length_success_rate,forbidden_count,avg_latency_sec,total_latency_sec,avg_generated_tokens,avg_tokens_per_sec,speed_score,judge_score,final_score,samples_per_min
0,exaone-7.8b,character_message_generation,10,2.330,1.00,0.00,0.0,1.00,0,6.889,68.893,NaN,NaN,3.496,1.000,1.600,8.709
1,exaone-7.8b,feed_post_generation,20,2.780,1.00,0.00,0.0,1.00,0,5.914,118.273,NaN,NaN,3.795,1.000,1.800,10.146
2,exaone-7.8b,planner_chatbot,20,2.200,1.00,0.00,0.0,1.00,0,5.782,115.640,NaN,NaN,4.922,1.000,1.540,10.377
3,exaone-7.8b,quest_generation,20,2.330,1.00,0.00,0.0,1.00,0,5.974,119.483,NaN,NaN,3.776,1.000,1.600,10.043
4,exaone-7.8b,todo_decomposition,20,2.200,1.00,0.00,0.0,1.00,0,5.731,114.627,NaN,NaN,3.852,1.000,1.540,10.469
5,gemma-3-4b-it,character_message_generation,10,2.330,1.00,0.00,0.0,1.00,0,0.352,3.522,NaN,NaN,5.000,1.000,1.600,170.372
6,gemma-3-4b-it,feed_post_generation,20,2.780,1.00,0.00,0.0,1.00,0,0.356,7.114,NaN,NaN,5.000,1.000,1.800,168.689
7,gemma-3-4b-it,planner_chatbot,20,2.200,1.00,0.00,0.0,1.00,0,0.357,7.135,NaN,NaN,5.000,1.000,1.540,168.176
8,gemma-3-4b-it,quest_generation,20,2.330,1.00,0.00,0.0,1.00,0,0.356,7.128,NaN,NaN,5.000,1.000,1.600,168.338
9,gemma-3-4b-it,todo_decomposition,20,2.200,1.00,0.00,0.0,1.00,0,0.351,7.024,NaN,NaN,5.000,1.000,1.540,170.853


In [20]:

if not SUMMARY_DF.empty:
    overall = SUMMARY_DF.groupby("model_key").agg(
        tasks=("task", "count"),
        auto_score=("auto_score", "mean"),
        json_parse_success_rate=("json_parse_success_rate", "mean"),
        schema_success_rate=("schema_success_rate", "mean"),
        avg_latency_sec=("avg_latency_sec", "mean"),
        total_latency_sec=("total_latency_sec", "sum"),
        avg_generated_tokens=("avg_generated_tokens", "mean"),
        avg_tokens_per_sec=("avg_tokens_per_sec", "mean"),
        speed_score=("speed_score", "mean"),
        samples_per_min=("samples_per_min", "mean"),
        judge_score=("judge_score", "mean"),
        final_score=("final_score", "mean"),
    ).round(3).reset_index()
    display(overall)

    try:
        ax = overall.plot.bar(x="model_key", y="auto_score", figsize=(10, 4), legend=False, title="모델별 평균 자동 점수")
        ax.set_ylabel("score")
    except Exception as exc:
        print("시각화 생략:", exc)
else:
    print("요약할 결과가 없습니다.")


,model_key,tasks,auto_score,json_parse_success_rate,schema_success_rate,avg_latency_sec,total_latency_sec,avg_generated_tokens,avg_tokens_per_sec,speed_score,samples_per_min,judge_score,final_score
0,exaone-7.8b,5,2.368,1.00,0.00,6.058,536.916,NaN,NaN,3.968,9.949,1.000,1.616
1,gemma-3-4b-it,5,2.368,1.00,0.00,0.354,31.923,NaN,NaN,5.000,169.286,1.000,1.616
2,midm-mini,5,4.216,0.95,0.95,1.844,175.648,56.01,30.145,4.934,39.598,3.337,3.733
3,phi-4-mini,1,2.200,1.00,0.00,3.115,3.115,NaN,NaN,4.660,19.263,1.000,1.540
4,qwen-7b,5,4.349,0.90,0.90,2.370,227.240,109.86,45.551,4.955,41.185,3.033,3.625


시각화 생략: matplotlib is required for plotting when the default backend "matplotlib" is selected.


## 15. Failure Case Review

In [22]:

FAILURE_CASES = collect_failure_cases(RAW_RECORDS)
print(f"failure cases: {len(FAILURE_CASES)}")

if FAILURE_CASES:
    pd.set_option("display.max_colwidth", None)
    failure_df = pd.DataFrame([
        {
            "model_key": f["model_key"],
            "task": f["task"],
            "sample_id": f["sample_id"],
            "auto_score": f["auto_eval"].get("auto_score"),
            "parse_error": f.get("parse_error"),
            "error": f.get("error"),
            "raw_output": f.get("raw_output", "")[:500],
        }
        for f in FAILURE_CASES
    ])
    display(failure_df)
    print("\n첫 번째 실패 원문 error:")
    print(FAILURE_CASES[0].get("error"))
else:
    print("자동 기준상 주요 실패 사례가 없습니다.")


failure cases: 227


model_key                          task         sample_id  \
0        midm-mini            todo_decomposition   TODO-SAMPLE-003   
1        midm-mini            todo_decomposition   TODO-SAMPLE-013   
2        midm-mini               planner_chatbot   PLAN-SAMPLE-007   
3        midm-mini              quest_generation  QUEST-SAMPLE-001   
4        midm-mini              quest_generation  QUEST-SAMPLE-002   
..             ...                           ...               ...   
222  gemma-3-4b-it  character_message_generation    MSG-SAMPLE-007   
223  gemma-3-4b-it  character_message_generation    MSG-SAMPLE-008   
224  gemma-3-4b-it  character_message_generation    MSG-SAMPLE-009   
225  gemma-3-4b-it  character_message_generation    MSG-SAMPLE-010   
226     phi-4-mini            todo_decomposition   TODO-SAMPLE-001   

     auto_score                                            parse_error  \
0          2.60   Expecting ',' delimiter: line 20 column 6 (char 322)   
1          2.60   Expecting ',' delimiter: line 20 column 6 (char 322)   
2          2.20  Expecting ',' delimiter: line 60 column 6 (char 1393)   
3          4.18                                                    NaN   
4          4.11                                                    NaN   
..          ...                                                    ...   
222        2.33                                                    NaN   
223        2.33                                                    NaN   
224        2.33                                                    NaN   
225        2.33                                                    NaN   
226        2.20                                                    NaN   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       


첫 번째 실패 원문 error:
None


## 16. Prompt / Hyperparameter Tuning Notes

모든 점수는 1~5점 기준입니다. 품질이 낮게 나온 경우 아래 순서로 조정하세요.

1. JSON 파싱 실패가 많으면 프롬프트에 `JSON으로만 응답`, `마크다운 코드블록 금지`, 출력 예시를 더 강하게 둡니다.
2. TODO/Planner처럼 구조가 중요한 기능은 `temperature`를 `0.0~0.2`로 낮춥니다.
3. 퀘스트/피드/말투처럼 감성 다양성이 필요한 기능은 `temperature`를 `0.5~0.8` 범위에서 비교합니다.
4. 반복이 많으면 `repetition_penalty`를 조금 올리거나 few-shot 예시를 다양하게 바꿉니다.
5. 퀘스트/피드에서 사용자 TODO가 노출되면 금지 조건을 프롬프트 앞쪽으로 옮기고 후처리 차단 규칙을 추가합니다.
6. GPT-5.5 심판 평가는 비용이 있으므로 후보를 좁힌 뒤 `JUDGE_CONFIG["enabled"] = True`로 켜는 것을 권장합니다.